# Lekcija 01 - Uvod v AI agente

Dobrodošli na prvi lekciji v tečaju **AI agenti za začetnike**!

**AI agent** je program, ki uporablja velik jezikovni model (LLM) kot svoj mehanizem sklepanja in lahko sprejema *dejanja* v resničnem svetu — kliče API-je, poizveduje baze podatkov ali izvršuje kodo — z namenom, da doseže cilj v imenu uporabnika.

V tej beležnici boste zgradili svoj prvi agent: **potovalni agent**, ki svetuje počitniške destinacije. Med potjo se boste naučili, kako:

1. Povezati se s storitvijo Azure AI Foundry Agent z uporabo **Microsoft Agent Framework**.
2. Agentu dati **orodje** — preprosto Python funkcijo, ki jo lahko kliče.
3. Zagnati agenta in pregledati njegov odgovor.
4. Tokovno prejemati agentov odgovor znak-po-znaku.


## Namestitev

Pred zagonom te zvezka se prepričajte, da imate:

1. **Projekt Azure AI Foundry** z nameščenim klepetalnim modelom (npr. `gpt-4o-mini`).
2. **Prijavljeni z Azure CLI** — zaženite `az login` v svojem terminalu.
3. **Nastavljene zahtevane sistemske spremenljivke:**
   - `AZURE_AI_PROJECT_ENDPOINT` — končna točka vašega projekta Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — ime vašega nameščenega modela.

Spodnja celica namesti potrebne Python pakete.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Ustvarjanje vašega prvega agenta

Agent potrebuje dve stvari:

- **Navodila**, ki mu povedo *kdo je* in *kako se naj obnaša* (sistemski poziv).
- **Orodja** — Python funkcije, okrašene z `@tool`, ki jih agent lahko pokliče za pridobivanje informacij ali izvajanje dejanj.

Spodaj definiramo preprosto orodje, ki vrne seznam priljubljenih počitniških destinacij. Agent bo uporabil to orodje, ko bo uporabnik prosil za potovalna priporočila.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Pretakanje odgovorov

Za bolj interaktivno izkušnjo lahko **pretakate** odgovor agenta. Namesto da bi čakali na celoten odgovor, agent sproti posreduje besedilne koščke, ko jih generira. To je še posebej uporabno v klepetalnih vmesnikih, kjer želite izpis prikazati v realnem času.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Povzetek

V tej lekciji ste se naučili, kako:

- **Ustvariti ponudnika**, ki se poveže z Azure AI Foundry Agent Service prek `AzureAIProjectAgentProvider`.
- **Določiti orodje** z uporabo dekoratorja `@tool`, da lahko agent kliče vaše Python funkcije.
- **Zagnati agenta** z uporabniškim sporočilom in izpisati njegov odgovor.
- **Pretakati odgovore** za izhod v realnem času.

V naslednji lekciji bomo poglobljeno raziskali agentske okvirje in se naučili, kako agentom zagotoviti močnejša orodja in večstopenjske sposobnosti sklepanja.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali nepravilnosti. Izvirni dokument v izvorni jezik je treba šteti za avtoritativni vir. Za ključne informacije priporočamo strokovni človeški prevod. Nismo odgovorni za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
